In [1]:
import os
import time
import joblib
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression

from Preprocessing_Pipeline import preprocess_english

c:\Users\Softlaptop\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
c:\Users\Softlaptop\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (
[2026-08-22 15:14:33,929 - farasapy_logger - WARNING]: Be careful with large lines as they may break on interactive mode. You may switch to Standalone mode for such cases.


In [2]:
df = pd.read_csv("../Datasets/MovieReviewTrainingDatabase.csv")
eng_df = df.copy()
eng_df.head(10)

,sentiment,review
0,Positive,With all this stuff going down at the moment w...
1,Positive,'The Classic War of the Worlds' by Timothy Hin...
2,Negative,The film starts with a manager (Nicholas Bell)...
3,Negative,It must be assumed that those who praised this...
4,Positive,Superbly trashy and wondrously unpretentious 8...
5,Positive,I dont know why people think this is such a ba...
6,Negative,"This movie could have been very good, but come..."
7,Negative,I watched this video at a friend's house. I'm ...
8,Negative,"A friend of mine bought this film for £1, and ..."
9,Positive,This movie is full of references. Like 'Mad ...


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   sentiment  25000 non-null  str  
 1   review     25000 non-null  str  
dtypes: str(2)
memory usage: 31.7 MB


In [4]:
eng_df['sentiment'].value_counts()

sentiment
Positive    12500
Negative    12500
Name: count, dtype: int64

In [5]:
eng_df['review'].duplicated().sum()

96

In [6]:
eng_df = eng_df.drop_duplicates()

In [7]:
X = eng_df['review']

y = eng_df['sentiment']
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)


In [8]:
X_train_clean = X_train.apply(preprocess_english)

X_test_clean = X_test.apply( preprocess_english)

In [9]:
print("\nCreating TF-IDF features...")

vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True,
    max_features=100000
)



Creating TF-IDF features...


In [10]:
X_train_tf = vectorizer.fit_transform(X_train_clean)

X_test_tf = vectorizer.transform(X_test_clean)

In [11]:
models = {
    "MultinomialNB": MultinomialNB(),
    "LinearSVC": LinearSVC(random_state=42),
    "LogisticRegression": LogisticRegression( max_iter=1000, random_state=42 ),
}

results = []
fitted_models = {}

for name, model in models.items():
    start_train = time.time()
    model.fit(X_train_tf, y_train)
    train_time = time.time() - start_train

    start_pred = time.time()
    val_preds = model.predict(X_test_tf)
    predict_time = time.time() - start_pred

    acc = accuracy_score(y_test, val_preds)
  

    fitted_models[name] = model
    results.append({
        "model": name,
        "val_accuracy": acc,
        "train_time_sec": train_time,
        "predict_time_sec": predict_time,
    })

result = pd.DataFrame(results).sort_values("val_accuracy", ascending=False)
result

,model,val_accuracy,train_time_sec,predict_time_sec
1,LinearSVC,0.904437,0.752565,0.006987
2,LogisticRegression,0.899819,3.339766,0.004999
0,MultinomialNB,0.883959,0.134254,0.010968


In [12]:
best_model_name = result.iloc[0]["model"]
best_model = fitted_models[best_model_name]

In [13]:
y_pred = best_model.predict( X_test_tf)

In [14]:
accuracy = accuracy_score( y_test,val_preds)
print("\n" + "=" * 60)
print("English SENTIMENT MODEL RESULTS")
print("=" * 60)

print(f"\nAccuracy: {accuracy:.4f}")

print("\nClassification Report:")

print( classification_report(  y_test,y_pred))

print("\nConfusion Matrix:")

print( confusion_matrix(y_test, y_pred))


English SENTIMENT MODEL RESULTS

Accuracy: 0.8998

Classification Report:
              precision    recall  f1-score   support

    Negative       0.91      0.90      0.90      2486
    Positive       0.90      0.91      0.91      2495

    accuracy                           0.90      4981
   macro avg       0.90      0.90      0.90      4981
weighted avg       0.90      0.90      0.90      4981


Confusion Matrix:
[[2235  251]
 [ 225 2270]]


In [15]:
joblib.dump(model, 'English_Model_Weights.pkl')
joblib.dump(vectorizer, 'English_Model_Vectorizer.pkl')
print("Best model saved to English_Model_Weights.pkl")

Best model saved to English_Model_Weights.pkl
